# Wikipedia の記事本文を API で取得する

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/01_data_io/wikipedia_api.ipynb)

NLP クックブック 01-02 ／ データの収集と入出力

特定の Wikipedia 記事の本文を、HTML を解析せずにプレーンテキストで取得したい。

In [1]:
import requests

API = "https://ja.wikipedia.org/w/api.php"
HEADERS = {"User-Agent": "nlp-cookbook-example/0.1 (educational notebook)"}   # 利用者を識別できる User-Agent を付けるのがルール

def fetch(title):
    params = {"action": "query", "prop": "extracts", "explaintext": 1, "titles": title, "format": "json", "redirects": 1}
    pages = requests.get(API, params=params, headers=HEADERS, timeout=30).json()["query"]["pages"]
    page = next(iter(pages.values()))
    return page.get("extract", "")

text = fetch("自然言語処理")
print(len(text), "文字")
print(text[:300])

4422 文字
自然言語処理（しぜんげんごしょり、英語: Natural language processing、略称：NLP）は、人間が日常的に使っている自然言語をコンピュータに処理させる一連の技術であり、人工知能と言語学の一分野である。「計算言語学」（computational linguistics）との類似もあるが、自然言語処理は工学的な視点からの言語処理をさすのに対して、計算言語学は言語学的視点を重視する手法をさす事が多い。データベース内の情報を自然言語に変換したり、自然言語の文章をより形式的な（コンピュータが理解しやすい）表現に変換するといった処理が含まれる。応用例としては機械翻訳や仮名漢字変換が


In [2]:
# キーワードで記事を検索する
def search(query, limit=5):
    params = {"action": "query", "list": "search", "srsearch": query, "srlimit": limit, "format": "json"}
    return [hit["title"] for hit in requests.get(API, params=params, headers=HEADERS, timeout=30).json()["query"]["search"]]

print(search("機械翻訳 ニューラル"))

['翻訳', '機械翻訳', 'ニューラルネットワーク (機械学習)', 'ニューラル機械翻訳', 'Google 翻訳']


- 数件〜数百件ならこの方法で。連続して呼ぶときは間隔を空ける (`time.sleep(1)`)。短時間に大量に呼ぶと 429 (Too Many Requests) で拒否される。
- 全記事が必要なら API を叩かずに、ダンプか Hugging Face の `wikimedia/wikipedia` データセットを使う（Transformers の基本操作の章のストリーミングのレシピを参照）。
- 本文のライセンスは CC BY-SA。再配布するときは出典の表示と同一ライセンスでの公開が必要。